# Data Science Pipeline: Product Recommendation

This notebook follows a complete workflow: business understanding, data assessment, feature engineering, a mixed-type machine-learning pipeline, hyperparameter tuning, evaluation, and inference.

## 1. Import Libraries and Load Data

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

data = pd.read_csv("data/reviews.csv")
print("Rows and columns:", data.shape)
data.head(3)

Rows and columns: (18442, 9)


,Clothing ID,Age,Title,Review Text,Positive Feedback Count,Division Name,Department Name,Class Name,Recommended IND
0,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,0,General,Dresses,Dresses,0
1,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",0,General Petite,Bottoms,Pants,1
2,847,47,Flattering shirt,This shirt is very flattering to all due to th...,6,General,Tops,Blouses,1


## 2. Assess, Clean, and Engineer Features

The target is `Recommended IND`. Missing review text is converted to an empty string, while missing numeric and categorical values are imputed inside the pipeline. Review length is created as a useful text-derived numeric feature.

In [2]:
data = data.dropna(subset=["Recommended IND"]).copy()
data["Review Text"] = data["Review Text"].fillna("")
data["Title"] = data["Title"].fillna("")
data["Review Length"] = data["Review Text"].str.len()
print("Missing values before pipeline imputation:")
print(data.isna().sum())
print("Target distribution:")
print(data["Recommended IND"].value_counts(normalize=True).rename("proportion"))

Missing values before pipeline imputation:
Clothing ID                0
Age                        0
Title                      0
Review Text                0
Positive Feedback Count    0
Division Name              0
Department Name            0
Class Name                 0
Recommended IND            0
Review Length              0
dtype: int64
Target distribution:
Recommended IND
1    0.816235
0    0.183765
Name: proportion, dtype: float64


## 3. Split Data Before Training

The test set is held out before fitting any transformation, preventing information leakage.

In [3]:
feature_columns = ["Review Text", "Title", "Age", "Positive Feedback Count", "Division Name", "Department Name", "Class Name", "Review Length"]
X = data[feature_columns]
y = data["Recommended IND"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

(14753, 8) (3689, 8)


## 4. Build the Mixed-Type Pipeline

TF-IDF normalizes and tokenizes review text, one-hot encoding handles categories, and imputation/scaling handles numeric inputs. All preprocessing remains inside the same pipeline as the classifier.

In [4]:
def to_1d(values):
    """Convert a selected text column to a one-dimensional string array."""
    if hasattr(values, "to_numpy"):
        values = values.to_numpy()
    return np.asarray(values).reshape(-1).astype(str)

numeric_features = ["Age", "Positive Feedback Count", "Review Length"]
categorical_features = ["Division Name", "Department Name", "Class Name"]
text_pipeline = Pipeline([
    ("to_1d", FunctionTransformer(to_1d, validate=False)),
    ("tfidf", TfidfVectorizer(lowercase=True, strip_accents="unicode", ngram_range=(1, 2), min_df=2, max_features=12000)),
])
numeric_pipeline = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
categorical_pipeline = Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
    ("review_text", text_pipeline, "Review Text"),
    ("title", text_pipeline, "Title"),
])
pipeline = Pipeline([("preprocess", preprocessor), ("model", LogisticRegression(max_iter=400, class_weight="balanced", solver="liblinear"))])
pipeline

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['Age',
                                                   'Positive Feedback Count',
                                                   'Review Length']),
                                                 ('categorical',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Divi...
                                                                                   ngram_range=(1,
                                                                                                2),
                                                                                   strip_accents='unicode'))]),
                                                  'Review Text'),
                                                 ('title',
                                                  Pipeline(steps=[('to_1d',
                                                                   FunctionTransformer(func=<function to_1d at 0x0000022B6ACF4220>)),
                                                                  ('tfidf',
                                                                   TfidfVectorizer(max_features=12000,
                                                                                   min_df=2,
                                                                                   ngram_range=(1,
                                                                                                2),
                                                                                   strip_accents='unicode'))]),
                                                  'Title')])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=400,
                                    solver='liblinear'))])

## 5. Fine-Tune and Train

In [5]:
search = GridSearchCV(pipeline, {"model__C": [0.5, 1.0]}, cv=3, scoring="f1", n_jobs=1, return_train_score=False)
search.fit(X_train, y_train)
model = search.best_estimator_
print("Best parameters:", search.best_params_)

Best parameters: {'model__C': 1.0}


## 6. Evaluate on Unseen Test Data

In [6]:
predictions = model.predict(X_test)
metrics = {
    "accuracy": accuracy_score(y_test, predictions),
    "precision": precision_score(y_test, predictions),
    "recall": recall_score(y_test, predictions),
    "f1": f1_score(y_test, predictions),
}
print(metrics)
print(classification_report(y_test, predictions, target_names=["does not recommend", "recommends"]))

{'accuracy': 0.9053944158308485, 'precision': np.float64(0.9736654804270463), 'recall': np.float64(0.9086682165393557), 'f1': np.float64(0.9400446658649717)}
                    precision    recall  f1-score   support

does not recommend       0.69      0.89      0.78       678
        recommends       0.97      0.91      0.94      3011

          accuracy                           0.91      3689
         macro avg       0.83      0.90      0.86      3689
      weighted avg       0.92      0.91      0.91      3689



## 7. Inference on a New Review

The same fitted object performs preprocessing and prediction for new observations.

In [7]:
new_review = pd.DataFrame([{
    "Review Text": "The fabric is comfortable and the fit is excellent.",
    "Title": "Great quality",
    "Age": 35,
    "Positive Feedback Count": 2,
    "Division Name": "General",
    "Department Name": "Dresses",
    "Class Name": "Casual",
    "Review Length": 50,
}])
print("Predicted recommendation:", int(model.predict(new_review)[0]))
print("Recommendation probability:", round(float(model.predict_proba(new_review)[0, 1]), 3))

Predicted recommendation: 1
Recommendation probability: 0.932


## Conclusion

The final model uses a single reproducible pipeline, handles all required data types and missing values, uses TF-IDF NLP features, is tuned with cross-validation, and is evaluated only after training on the held-out test set.